* Name: Shaun Russell
* L-number: L00181248
* Date: 15/02/2026

## Automated Download of swellwesad dataset

In [5]:
import sys
# 1. Fix for Python 3.13 'cgi' removal
try:
    import cgi
except ImportError:
    import html
    sys.modules['cgi'] = html 

import opendatasets as od
import pandas as pd
import os

# 2. Download - it will ask for your Kaggle Username/Key
dataset_url = "https://www.kaggle.com/datasets/jfraisa/swellwesad-hrv-data"

try:
    od.download(dataset_url)
    print("Download Complete!")
except Exception as e:
    print(f"If download failed, ensure kaggle.json is in your .kaggle folder. Error: {e}")

# 3. Check the folder name (opendatasets usually names it after the slug)
folder_name = 'swell-wesad-hrv-data'
if os.path.exists(folder_name):
    print(f"Files found in: {os.listdir(folder_name)}")



Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:

  shaunrussell2706


Your Kaggle Key:

  ········


Dataset URL: https://www.kaggle.com/datasets/jfraisa/swellwesad-hrv-data


100%|███████████████████████████████████████████████████████████████████████████████| 246M/246M [00:07<00:00, 35.2MB/s]



Download Complete!


## finding the folder

In [12]:
import os

# 1. List all folders in your current directory
print("Current folders:", [d for d in os.listdir() if os.path.isdir(d)])

# 2. Automatically find the Kaggle folder (it usually starts with 'swell')
possible_folders = [d for d in os.listdir() if 'swell' in d.lower()]

if possible_folders:
    folder = possible_folders[0]
    print(f"\nFound the folder: {folder}")
    print("Files inside that folder:", os.listdir(folder))
else:
    print("\nNo folder found. Try running the download cell again.")


Current folders: ['.ipynb_checkpoints', 'swellwesad-hrv-data']

Found the folder: swellwesad-hrv-data
Files inside that folder: ['HRV Dataset']


## display dataset contents

In [13]:
import pandas as pd
import os

# 1. Define the folder structure correctly
base_folder = 'swellwesad-hrv-data'
sub_folder = 'HRV Dataset'

# 2. List the files inside 'HRV Dataset' to find the exact CSV name
inner_path = os.path.join(base_folder, sub_folder)
print("Files inside HRV Dataset:", os.listdir(inner_path))

# 3. Use the first CSV file found in that folder
# (It's likely called 'hrv_dataset.csv' or similar)
files = [f for f in os.listdir(inner_path) if f.endswith('.csv')]

if files:
    csv_file = os.path.join(inner_path, files[0])
    print(f"\nLoading: {csv_file}")
    
    # 4. Load the preview
    df_preview = pd.read_csv(csv_file, nrows=5)
    print("\n--- Dataset Columns ---")
    print(df_preview.columns.tolist())
else:
    print("\nNo CSV files found in that subfolder!")



Files inside HRV Dataset: ['combined-swell-classification-hrv-test-dataset.csv', 'combined-swell-classification-hrv-train-dataset.csv', 'wesad-classification-hrv-test-dataset.csv', 'wesad-classification-hrv-train-dataset.csv']

Loading: swellwesad-hrv-data\HRV Dataset\combined-swell-classification-hrv-test-dataset.csv

--- Dataset Columns ---
['MEAN_RR', 'MEDIAN_RR', 'SDRR', 'RMSSD', 'SDSD', 'SDRR_RMSSD', 'HR', 'pNN25', 'pNN50', 'SD1', 'SD2', 'KURT', 'SKEW', 'MEAN_REL_RR', 'MEDIAN_REL_RR', 'SDRR_REL_RR', 'RMSSD_REL_RR', 'SDSD_REL_RR', 'SDRR_RMSSD_REL_RR', 'KURT_REL_RR', 'SKEW_REL_RR', 'VLF', 'VLF_PCT', 'LF', 'LF_PCT', 'LF_NU', 'HF', 'HF_PCT', 'HF_NU', 'TP', 'LF_HF', 'HF_LF', 'sampen', 'higuci', 'condition', 'subject_id', 'MEAN_RR_LOG', 'MEAN_RR_SQRT', 'TP_SQRT', 'MEDIAN_REL_RR_LOG', 'RMSSD_REL_RR_LOG', 'SDSD_REL_RR_LOG', 'VLF_LOG', 'LF_LOG', 'HF_LOG', 'TP_LOG', 'LF_HF_LOG', 'RMSSD_LOG', 'SDRR_RMSSD_LOG', 'pNN25_LOG', 'pNN50_LOG', 'SD1_LOG', 'KURT_YEO_JONSON', 'SKEW_YEO_JONSON', 'MEAN_R

## Train the Model using random forest

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import pandas as pd
import os

# STEP 1: AUTOMATICALLY FIND THE CSV FILE 
def find_csv():
    for root, dirs, files in os.walk("."):
        for file in files:
            if file.endswith(".csv") and "hrv" in file.lower():
                return os.path.join(root, file)
    return None

file_path = find_csv()

if file_path is None:
    print("❌ ERROR: Still can't find a CSV file with 'hrv' in the name.")
    print("Check if you unzipped the download!")
else:
    print(f"✅ Found File at: {file_path}")
    
    # STEP 2: TRAINING LOGIC
    X_list = []
    y_list = []
    chunk_size = 50000 

    print("Loading data in chunks...")
    
    # Read the file to get column names first
    header = pd.read_csv(file_path, nrows=1)
    header.columns = header.columns.str.strip()
    
    # Determine the correct label column name
    target_col = 'label' if 'label' in header.columns else 'condition'
    features = ['HR', 'RMSSD', 'SDRR', 'MEAN_RR', 'MEDIAN_RR']

    # Process chunks
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        chunk.columns = chunk.columns.str.strip()
        
        # Drop rows with missing values in our target columns
        sample = chunk.dropna(subset=features + [target_col]).sample(frac=0.2, random_state=42)
        
        X_list.append(sample[features])
        y_list.append(sample[target_col])

    # Combine
    X = pd.concat(X_list)
    y = pd.concat(y_list)

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    print(f"Training Random Forest on {len(X_train)} rows...")
    model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)

    # Save
    joblib.dump(model, 'burnout_model.pkl')
    joblib.dump(scaler, 'scaler.pkl')

    print("🚀 SUCCESS! Model trained and saved as 'burnout_model.pkl'.")


✅ Found File at: .\swellwesad-hrv-data\HRV Dataset\combined-swell-classification-hrv-test-dataset.csv
Loading data in chunks...
Training Random Forest on 6266 rows...
🚀 SUCCESS! Model trained and saved as 'burnout_model.pkl'.


# Tests

## Testing the model using synthetic data

In [1]:
import joblib
import pandas as pd
import numpy as np

# 1. Load the "Brain"
model = joblib.load('burnout_model.pkl')
scaler = joblib.load('scaler.pkl')

# 2. Define the exact features we used to train (Version 1.0)
features = ['HR', 'RMSSD', 'SDRR', 'MEAN_RR', 'MEDIAN_RR']

# 3. Create a "Fake" Row of Garmin Data using a DataFrame
# This specifically fixes the UserWarning!
test_values = [[105.0, 12.5, 18.0, 580.0, 575.0]] # Stressful numbers
test_df = pd.DataFrame(test_values, columns=features)

# 4. Scale and Predict
test_scaled = scaler.transform(test_df)
prediction = model.predict(test_scaled)
probability = model.predict_proba(test_scaled)

# 5. Display the result
status = "STRESSED / BURNOUT SIGNS" if prediction[0] == 1 else "NORMAL / RECOVERED"
confidence = np.max(probability) * 100

print(f"--- AI INFERENCE RESULT ---")
print(f"Input: {test_values[0]}")
print(f"Prediction: {status}")
print(f"Confidence Score: {confidence:.2f}%")


--- AI INFERENCE RESULT ---
Input: [105.0, 12.5, 18.0, 580.0, 575.0]
Prediction: NORMAL / RECOVERED
Confidence Score: 47.00%


## Testing the model on personal data (trying to find most recent day)

In [2]:
import pandas as pd
import joblib
import numpy as np

# 1. Load the "Brain"
model = joblib.load('burnout_model.pkl')
scaler = joblib.load('scaler.pkl')

# 2. Load your Garmin CSV
df_garmin = pd.read_csv('garmin_burnout_data.csv') 

# CLEANING STEP: Remove hidden spaces and fix capitalization ---
df_garmin.columns = df_garmin.columns.str.strip().str.lower()
# If your column was 'Date', it is now 'date'

# 3. Filter for the specific date: 16/02/2026
target_date = "16-02-2026"
day_data = df_garmin[df_garmin['date'].astype(str).str.contains(target_date)].copy()

if day_data.empty:
    print(f"Still can't find {target_date}. Your dates look like: {df_garmin['date'].head().tolist()}")
else:
    # 4. MAP THE DATA FROM YOUR IMAGE
    # Looking at your image: [Date, 43, 90, 70, 53]
    # We assume: 43 = avg_stress, 90 = max_stress, 70 = rest_hr, 53 = hrv_avg
    
    hr_val = day_data['rest_hr'].values[0]
    hrv_val = day_data['hrv_avg'].values[0]

    # 'Common Language' translation for the AI
    mapping = {
        'HR': hr_val,
        'RMSSD': hrv_val,
        'SDRR': hrv_val * 1.1,         # Estimate based on RMSSD
        'MEAN_RR': 60000 / hr_val,    # Convert BPM to milliseconds
        'MEDIAN_RR': 60000 / hr_val
    }

    # 5. Create Input for AI
    features = ['HR', 'RMSSD', 'SDRR', 'MEAN_RR', 'MEDIAN_RR']
    input_df = pd.DataFrame([mapping], columns=features)

    # 6. Scale and Predict
    input_scaled = scaler.transform(input_df)
    prediction = model.predict(input_scaled)
    probability = model.predict_proba(input_scaled)

    # 7. RESULTS
    status = "BURNOUT RISK DETECTED" if prediction == 1 else "✅ NORMAL / RECOVERED"
    score = np.max(probability) * 100

    print(f"--- Results for {target_date} ---")
    print(f"Physiology: HR {hr_val} | HRV {hrv_val}")
    print(f"AI Prediction: {status}")
    print(f"Model Confidence: {score:.2f}%")


❌ Still can't find 16-02-2026. Your dates look like: ['2026-01-22', '2026-01-23', '2026-01-24', '2026-01-25', '2026-01-26']


In [3]:
import pandas as pd
import joblib
import numpy as np
import datetime as dt

# Load model + scaler
model = joblib.load('burnout_model.pkl')
scaler = joblib.load('scaler.pkl')

# Load Garmin CSV
df_garmin = pd.read_csv('garmin_burnout_data.csv')
df_garmin.columns = df_garmin.columns.str.strip().str.lower()

# Parse dates cleanly
df_garmin['date'] = pd.to_datetime(df_garmin['date'], errors='coerce').dt.date

# Pick today's row (fallback to latest row if today missing)
today = dt.date.today()
day_data = df_garmin[df_garmin['date'] == today].copy()

if day_data.empty:
    day_data = df_garmin.dropna(subset=['date']).sort_values('date').tail(1).copy()
    print(f" No row for today ({today}). Using latest date: {day_data['date'].iloc[0]}")

# Grab values
hr_val = day_data['rest_hr'].iloc[0] if 'rest_hr' in day_data else np.nan
hrv_val = day_data['hrv_avg'].iloc[0] if 'hrv_avg' in day_data else np.nan

# If HRV is missing today, use most recent non-null HRV
if pd.isna(hrv_val) and 'hrv_avg' in df_garmin.columns:
    prev = df_garmin.dropna(subset=['hrv_avg']).sort_values('date').tail(1)
    if not prev.empty:
        hrv_val = prev['hrv_avg'].iloc[0]
        print(f"Today's HRV missing. Using most recent HRV: {hrv_val} from {prev['date'].iloc[0]}")

# Stop if we still can't run
if pd.isna(hr_val) or pd.isna(hrv_val):
    print(f"Not enough data to predict. rest_hr={hr_val}, hrv_avg={hrv_val}")
else:
    # Map into model features
    features = ['HR', 'RMSSD', 'SDRR', 'MEAN_RR', 'MEDIAN_RR']
    mapping = {
        'HR': float(hr_val),
        'RMSSD': float(hrv_val),
        'SDRR': float(hrv_val) * 1.1,
        'MEAN_RR': 60000.0 / float(hr_val),
        'MEDIAN_RR': 60000.0 / float(hr_val),
    }

    input_df = pd.DataFrame([mapping], columns=features)

    input_scaled = scaler.transform(input_df)
    pred = model.predict(input_scaled)[0]
    proba = model.predict_proba(input_scaled)[0].max() * 100

    status = "⚠️ BURNOUT RISK DETECTED" if pred == 1 else "✅ NORMAL / RECOVERED"
    print(f"--- Results for {day_data['date'].iloc[0]} ---")
    print(f"Physiology: Rest HR {hr_val} | HRV {hrv_val}")
    print(f"AI Prediction: {status}")
    print(f"Model Confidence: {proba:.2f}%")

⚠️ Today's HRV missing. Using most recent HRV: 42.0 from 2026-02-20
--- Results for 2026-02-23 ---
Physiology: Rest HR 72.0 | HRV 42.0
AI Prediction: ✅ NORMAL / RECOVERED
Model Confidence: 39.00%
